# AnimalPoseTracker Demo

Run a 10-epoch AnimalRTPose-N workflow on Kaggle: training, validation, test prediction from `.pt`, ONNX export, and test prediction from ONNX. Attach the dataset under Kaggle input `AnimalDatasets`; it is read directly from `/kaggle/input` without copying. Outputs are saved under `/kaggle/working/wux024`. Enable a GPU accelerator before running.

Set `RUN_MODE` in the configuration cell to `new` or `load`. In load mode, attach the trained checkpoint as a Kaggle input and set `CHECKPOINT_PATH` to its `/kaggle/input/.../best.pt` path.


In [ ]:
# Run this cell, choose a mode and checkpoint path, then click Apply.
# If the controls do not render in this Kaggle session, edit the fallback values below.
RUN_MODE = "new"
CHECKPOINT_PATH = ""

try:
    import ipywidgets as widgets
    from IPython.display import display

    mode_widget = widgets.Dropdown(
        options=[("New training project", "new"), ("Load checkpoint", "load")],
        value=RUN_MODE,
        description="Mode:",
        layout=widgets.Layout(width="360px"),
    )
    checkpoint_widget = widgets.Text(
        value=CHECKPOINT_PATH,
        description="Checkpoint:",
        placeholder="/kaggle/input/.../best.pt",
        layout=widgets.Layout(width="760px"),
    )
    apply_button = widgets.Button(description="Apply", button_style="primary")
    status = widgets.HTML()

    def apply_run_options(_=None):
        global RUN_MODE, CHECKPOINT_PATH
        RUN_MODE = mode_widget.value
        CHECKPOINT_PATH = checkpoint_widget.value.strip()
        status.value = f"<b>Applied:</b> {RUN_MODE} — continue with the next cell."
        if RUN_MODE == "load":
            status.value += " — checkpoint path set" if CHECKPOINT_PATH else " — enter a checkpoint path"

    apply_button.on_click(apply_run_options)
    display(widgets.VBox([mode_widget, checkpoint_widget, apply_button, status]))
    apply_run_options()
except ImportError:
    print("ipywidgets unavailable; use the RUN_MODE and CHECKPOINT_PATH fallback values above.")


In [ ]:
from datetime import datetime
from pathlib import Path

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
WORKSPACE_ROOT = Path("/kaggle/working/wux024")
REPO_DIR = WORKSPACE_ROOT / "AnimalPoseTracker"
PROJECTS_ROOT = WORKSPACE_ROOT
REPO_URL = "https://github.com/wux024/AnimalPoseTracker.git"
BRANCH = "preview"
KAGGLE_COLLECTION_NAME = "AnimalDatasets"
DATA_ROOT_OVERRIDE = None
RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S")

EPOCHS = 10
BATCH_SIZE = 4
IMAGE_SIZE = 640
NUM_WORKERS = 2
DEVICE = "0"
PREDICT_SPLIT = "test"
if RUN_MODE not in {"new", "load"}:
    raise ValueError("RUN_MODE must be 'new' or 'load'")


def normalise_name(value):
    return "".join(char for char in value.casefold() if char.isalnum())


collection_matches = [
    path for path in KAGGLE_INPUT_ROOT.iterdir()
    if path.is_dir() and normalise_name(KAGGLE_COLLECTION_NAME) in normalise_name(path.name)
]
if len(collection_matches) != 1:
    raise FileNotFoundError(
        f"Expected one Kaggle input matching {KAGGLE_COLLECTION_NAME!r}; "
        f"available inputs: {[path.name for path in KAGGLE_INPUT_ROOT.iterdir()]}"
    )
KAGGLE_COLLECTION_ROOT = collection_matches[0]
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)

BEST_CHECKPOINT = None
if RUN_MODE == "load":
    if not CHECKPOINT_PATH.strip():
        raise ValueError("Set CHECKPOINT_PATH to the attached /kaggle/input/.../best.pt file")
    BEST_CHECKPOINT = Path(CHECKPOINT_PATH).expanduser().resolve()
    if not BEST_CHECKPOINT.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {BEST_CHECKPOINT}")
    print("Loading checkpoint:", BEST_CHECKPOINT)
else:
    print("Mode: create a new project and train")
print("AnimalDatasets:", KAGGLE_COLLECTION_ROOT)
print("Workspace:", WORKSPACE_ROOT)

In [ ]:
import subprocess

if not (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
REPO_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True
).strip()
print("Repository:", REPO_DIR, REPO_COMMIT)

In [ ]:
%cd /kaggle/working/wux024/AnimalPoseTracker
%pip install -v --no-deps -e .
%pip install -q pycocotools onnx onnxsim onnxruntime

In [ ]:
import yaml

if DATA_ROOT_OVERRIDE is not None:
    DATA_SOURCE_ROOT = Path(DATA_ROOT_OVERRIDE).expanduser().resolve()
else:
    image_dirs = [
        path for path in KAGGLE_COLLECTION_ROOT.rglob("images")
        if path.is_dir()
        and all((path / split).is_dir() for split in ("train", "val", "test"))
    ]
    roots = sorted({path.parent.resolve() for path in image_dirs})
    trimouse_roots = [
        path for path in roots
        if "trimouse" in normalise_name(str(path))
    ]
    candidates = trimouse_roots or roots
    if len(candidates) != 1:
        raise RuntimeError(
            "Could not select one Trimouse dataset inside AnimalDatasets. "
            f"Candidates: {[str(path) for path in candidates]}. "
            "Set DATA_ROOT_OVERRIDE to the dataset folder containing images/."
        )
    DATA_SOURCE_ROOT = candidates[0]

DATA_YAML = yaml.safe_load(
    (REPO_DIR / "animalposetracker" / "cfg" / "datasets" / "trimouse.yaml")
    .read_text(encoding="utf-8")
)
DATA_YAML["path"] = str(DATA_SOURCE_ROOT)
if not DATA_YAML.get("annotation_format"):
    annotation_dir = DATA_SOURCE_ROOT / "annotations"
    annotation_files = [
        annotation_dir / f"{split}.json" for split in ("train", "val", "test")
    ] + [
        annotation_dir / f"person_keypoints_{split}2017.json"
        for split in ("train", "val", "test")
    ]
    DATA_YAML["annotation_format"] = (
        "coco" if any(path.is_file() for path in annotation_files) else "yolo"
    )
print("Trimouse input:", DATA_SOURCE_ROOT)
print("Annotation format:", DATA_YAML["annotation_format"])
print("Keypoint shape:", DATA_YAML["kpt_shape"])

In [ ]:
from animalposetracker.project import AnimalPoseTrackerProject

keypoint_count, keypoint_dims = DATA_YAML["kpt_shape"]
class_names = {int(key): str(value) for key, value in DATA_YAML["names"].items()}
project = AnimalPoseTrackerProject(
    local_path=PROJECTS_ROOT,
    project_name="trimouse",
    worker="kaggle",
    model_type="AnimalRTPose",
    model_scale="N",
    keypoints=keypoint_count,
    visible=(keypoint_dims == 3),
    classes=len(class_names),
    keypoints_name=DATA_YAML.get(
        "keypoint_names", [f"kpt_{index}" for index in range(keypoint_count)]
    ),
    skeleton=DATA_YAML.get("skeleton", []),
    kpt_oks_sigmas=DATA_YAML.get("kpt_oks_sigmas"),
    classes_name=[class_names[index] for index in sorted(class_names)],
    date=RUN_ID,
)
project.create_new_project()
project.dataset_config.update(DATA_YAML)
project.dataset_config["path"] = str(DATA_SOURCE_ROOT)
project.dataset_config["names"] = class_names
project.project_config.update({
    "classes": len(class_names),
    "classes_name": [class_names[index] for index in sorted(class_names)],
    "keypoints": keypoint_count,
    "keypoints_name": DATA_YAML.get(
        "keypoint_names", [f"kpt_{index}" for index in range(keypoint_count)]
    ),
    "visible": (keypoint_dims == 3),
    "skeleton": DATA_YAML.get("skeleton", []),
    "project_path": str(project.project_path),
})
project._sync_model_config_from_dataset()
project.other_config.update({
    "epochs": EPOCHS,
    "batch": BATCH_SIZE,
    "imgsz": IMAGE_SIZE,
    "workers": NUM_WORKERS,
    "device": DEVICE,
    "pretrained": RUN_MODE == "new",
    "val": True,
    "plots": True,
    "project": "runs",
    "name": "train",
    "exist_ok": False,
})
project._detect_pretrained()
project.save_configs("all")
PROJECT_DIR = project.project_path
print("Project:", PROJECT_DIR)
print("Pretrained weights:", project.other_config.get("pretrained"))

In [ ]:
import json
import os
import shlex
import subprocess
from pathlib import Path


def _format_cli_metrics(metrics):
    return ", ".join(
        f"{key}={value:.4g}"
        for key, value in metrics.items()
        if isinstance(value, (int, float))
    )


def _show_cli_line(line):
    text = line.rstrip()
    if not text:
        return
    try:
        event = json.loads(text)
    except json.JSONDecodeError:
        print(text, flush=True)
        return
    if not isinstance(event, dict) or "event" not in event:
        print(text, flush=True)
        return

    kind = event["event"]
    epoch = event.get("epoch")
    epochs = event.get("epochs")
    metrics = _format_cli_metrics(event.get("metrics") or {})
    if kind == "batch":
        position = f"epoch {epoch}/{epochs}, batch {event.get('step')}/{event.get('steps')}"
        print(f"  {position}" + (f" | {metrics}" if metrics else ""), flush=True)
    elif kind == "epoch_end":
        print(f"Epoch {epoch}/{epochs} complete" + (f" | {metrics}" if metrics else ""), flush=True)
    elif kind == "validation":
        print(f"Validation {epoch}/{epochs}" + (f" | {metrics}" if metrics else ""), flush=True)
    else:
        details = [event.get("message"), metrics]
        details = [str(value) for value in details if value]
        print(f"[{kind}] " + " | ".join(details), flush=True)


def run_cli(*arguments):
    command = ["animalpose-cli", *map(str, arguments)]
    print("$ " + " ".join(shlex.quote(value) for value in command), flush=True)
    environment = os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"
    process = subprocess.Popen(
        command,
        cwd=PROJECT_DIR,
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError("Could not read CLI output")
    for line in process.stdout:
        _show_cli_line(line)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)


os.chdir(PROJECT_DIR)
if RUN_MODE == "new":
    print("Starting training. Progress events will appear below.", flush=True)
    run_cli("train", "--config", "configs/other.yaml")
    checkpoint_candidates = sorted(
        Path("runs").glob("train*/weights/best.pt"),
        key=lambda path: path.stat().st_mtime,
    )
    if not checkpoint_candidates:
        raise FileNotFoundError("Training did not produce runs/train*/weights/best.pt")
    BEST_CHECKPOINT = checkpoint_candidates[-1].resolve()
else:
    print("Skipping training; reusing the selected checkpoint.", flush=True)
print("Best checkpoint:", BEST_CHECKPOINT)


In [ ]:
import json
from pathlib import Path

run_cli("val", "--config", "configs/other.yaml", "--weights", BEST_CHECKPOINT)
metric_candidates = sorted(
    Path("runs").glob("val*/metrics.json"),
    key=lambda path: path.stat().st_mtime,
)
if not metric_candidates:
    raise FileNotFoundError("Validation did not produce runs/val*/metrics.json")
VALIDATION_METRICS = metric_candidates[-1].resolve()
print("Validation metrics:", json.dumps(
    json.loads(VALIDATION_METRICS.read_text(encoding="utf-8")),
    indent=2,
    ensure_ascii=False,
))


In [ ]:
from pathlib import Path

run_cli(
    "predict", "--config", "configs/other.yaml",
    "--weights", BEST_CHECKPOINT, "--split", PREDICT_SPLIT,
)
prediction_candidates = sorted(
    Path("runs").glob("predict*/predictions.json"),
    key=lambda path: path.stat().st_mtime,
)
if not prediction_candidates:
    raise FileNotFoundError("Prediction did not produce runs/predict*/predictions.json")
PT_PREDICTIONS = prediction_candidates[-1].resolve()
print(".pt predictions:", PT_PREDICTIONS)


In [ ]:
from pathlib import Path

run_cli(
    "export", "--config", "configs/other.yaml",
    "--weights", BEST_CHECKPOINT, "--format", "onnx",
    "--output-dir", "runs/export",
)
onnx_candidates = sorted(
    Path("runs").glob("export*/best.onnx"),
    key=lambda path: path.stat().st_mtime,
)
if not onnx_candidates:
    raise FileNotFoundError("Export did not produce runs/export*/best.onnx")
ONNX_MODEL = onnx_candidates[-1].resolve()
run_cli(
    "predict", "--config", "configs/other.yaml",
    "--weights", ONNX_MODEL, "--split", PREDICT_SPLIT,
)
onnx_prediction_candidates = sorted(
    Path("runs").glob("predict*/predictions.json"),
    key=lambda path: path.stat().st_mtime,
)
if not onnx_prediction_candidates:
    raise FileNotFoundError("ONNX prediction did not produce runs/predict*/predictions.json")
ONNX_PREDICTIONS = onnx_prediction_candidates[-1].resolve()
print("ONNX model:", ONNX_MODEL)
print("ONNX predictions:", ONNX_PREDICTIONS)


In [ ]:
import json
from pathlib import Path
import yaml

project_path = Path(PROJECT_DIR).resolve()

def make_pretrained_paths_relative(value):
    if isinstance(value, dict):
        for key, item in list(value.items()):
            if key == "pretrained" and isinstance(item, str):
                candidate = Path(item).expanduser()
                if candidate.is_absolute():
                    try:
                        value[key] = str(candidate.resolve().relative_to(project_path))
                    except ValueError:
                        pass
            else:
                make_pretrained_paths_relative(item)
    elif isinstance(value, list):
        for item in value:
            make_pretrained_paths_relative(item)


dataset_path = project_path / "configs" / "dataset.yaml"
dataset_config = yaml.safe_load(dataset_path.read_text(encoding="utf-8"))
dataset_config["path"] = str(DATA_SOURCE_ROOT)
dataset_path.write_text(
    yaml.safe_dump(dataset_config, sort_keys=False, allow_unicode=True), encoding="utf-8"
)
other_path = project_path / "configs" / "other.yaml"
other_config = yaml.safe_load(other_path.read_text(encoding="utf-8"))
other_config["model"] = "configs/model.yaml"
other_config["data"] = "configs/dataset.yaml"
make_pretrained_paths_relative(other_config)
other_path.write_text(
    yaml.safe_dump(other_config, sort_keys=False, allow_unicode=True), encoding="utf-8"
)

def relative(path):
    try:
        return str(Path(path).resolve().relative_to(project_path))
    except ValueError:
        return str(path)

summary = {
    "repository": REPO_URL,
    "branch": BRANCH,
    "commit": REPO_COMMIT,
    "model": "AnimalRTPose-N",
    "dataset": str(DATA_SOURCE_ROOT),
    "annotation_format": DATA_YAML["annotation_format"],
    "run_mode": RUN_MODE,
    "epochs": EPOCHS,
    "batch": BATCH_SIZE,
    "best_checkpoint": relative(BEST_CHECKPOINT),
    "validation_metrics": relative(VALIDATION_METRICS),
    "pt_predictions": relative(PT_PREDICTIONS),
    "onnx_model": relative(ONNX_MODEL),
    "onnx_predictions": relative(ONNX_PREDICTIONS),
    "project": str(project_path),
}
summary_path = project_path / "notebook_run_summary.json"
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False) + chr(10), encoding="utf-8")
print("Saved project and outputs:", project_path)
print(json.dumps(summary, indent=2, ensure_ascii=False))